# Lab type: review
# Course: ML302 — Transformer Models & Fine-Tuning
# Lesson: Working with Hugging Face Transformers
# Task: The code below is correct and working. Read each section, run it, then answer the judgment questions in the markdown cells below each block.

In [ ]:
# Install dependencies (uncomment if running on Colab)
# !pip install transformers torch

from transformers import AutoTokenizer, AutoModelForSequenceClassification
import torch

tokenizer = AutoTokenizer.from_pretrained('distilbert-base-uncased')
model = AutoModelForSequenceClassification.from_pretrained('distilbert-base-uncased', num_labels=2)
model.eval()
print('Model loaded. Params:', sum(p.numel() for p in model.parameters()))

## Part 1: Attention Mask in Batch Inference

The cells below show single-input inference (correct) and batch inference — first with the missing attention mask (the common AI-generated pattern), then with the correct full-dict unpacking.

In [ ]:
# Single input — no padding needed
text_single = 'This model works well.'
inputs_single = tokenizer(text_single, return_tensors='pt')

print('Single input keys:', list(inputs_single.keys()))
print('attention_mask:', inputs_single['attention_mask'])

with torch.no_grad():
    out = model(**inputs_single)
print('Logits:', out.logits.numpy().round(4))

In [ ]:
# Batch input — padding is applied
texts = [
    'Short text.',
    'This is a much longer piece of text that requires padding to the batch maximum.',
]
inputs_batch = tokenizer(texts, padding=True, return_tensors='pt')

print('Batch input_ids shape:', inputs_batch['input_ids'].shape)
print('attention_mask:')
print(inputs_batch['attention_mask'])
print()
print('Row 0 (short, padded):',  inputs_batch['attention_mask'][0].tolist())
print('Row 1 (long, no pad):  ', inputs_batch['attention_mask'][1].tolist())

In [ ]:
# WRONG: pass only input_ids, discard attention_mask
with torch.no_grad():
    out_wrong = model(input_ids=inputs_batch['input_ids'])
print('Logits (no mask — padding attended to):', out_wrong.logits.numpy().round(4))

# CORRECT: pass the full dict so attention_mask is included
with torch.no_grad():
    out_correct = model(**inputs_batch)
print('Logits (with mask — correct):          ', out_correct.logits.numpy().round(4))

print()
print('Difference in row 0 logits (the padded sequence):')
print((out_correct.logits[0] - out_wrong.logits[0]).abs().numpy().round(4))

**Question 1:** Row 0 is the shorter text. Run the cell and record the difference between the correct and wrong logits for row 0. Row 1 is not padded — what difference do you expect for row 1, and why?

*(Write your answer here.)*

**Question 2:** A colleague argues: 'The attention weights on padding tokens are small, so the error is negligible.' Given that transformer representations are built by iterating through multiple attention layers, why does even a small nonzero weight on padding tokens compound across layers? Describe the effect on the final token representation.

*(Write your answer here.)*

## Part 2: Left-Padding for Decoder-Only Models

Decoder-only models (GPT-2, LLaMA) generate by reading the last token position's hidden state. Padding must be on the left so that position is always a real input token.

In [ ]:
# Demonstrate right-padding vs left-padding effect on last-token position
gpt2_tokenizer_right = AutoTokenizer.from_pretrained('gpt2')
gpt2_tokenizer_right.pad_token = gpt2_tokenizer_right.eos_token
gpt2_tokenizer_right.padding_side = 'right'  # default — wrong for decoder-only batch

gpt2_tokenizer_left = AutoTokenizer.from_pretrained('gpt2')
gpt2_tokenizer_left.pad_token = gpt2_tokenizer_left.eos_token
gpt2_tokenizer_left.padding_side = 'left'   # correct for decoder-only batch

texts = ['Hello', 'Hello world, how are you?']

right_enc = gpt2_tokenizer_right(texts, padding=True, return_tensors='pt')
left_enc  = gpt2_tokenizer_left(texts,  padding=True, return_tensors='pt')

print('Right-padding — input_ids:')
for row in right_enc['input_ids']:
    decoded = [gpt2_tokenizer_right.decode([t]) for t in row.tolist()]
    print(' ', decoded)
print()
print('Left-padding — input_ids:')
for row in left_enc['input_ids']:
    decoded = [gpt2_tokenizer_left.decode([t]) for t in row.tolist()]
    print(' ', decoded)

print()
print('With right-padding, last token of row 0 is:', gpt2_tokenizer_right.decode([right_enc['input_ids'][0, -1].item()]))
print('With left-padding,  last token of row 0 is:', gpt2_tokenizer_left.decode([left_enc['input_ids'][0, -1].item()]))

**Question 3:** With right-padding, the last token for the shorter sequence ('Hello') is a padding token. A GPT-2 model generates the next token from the last position's hidden state. In one sentence, describe exactly what the model is predicting from when padding is on the right.

*(Write your answer here.)*

**Question 4:** Left-padding ensures the last position is always a real token. However, this means attention positions differ between the two sequences in the batch. Positional embeddings in GPT-2 are learned and absolute — position 0 always means 'start of sequence'. With left-padding, what does position 0 mean for the shorter sequence in a batch? Is this a concern, and why?

*(Write your answer here.)*

## Part 3: Model Card Checklist

Before deploying any checkpoint from the Hugging Face Hub, you should check five things from the model card.

In [ ]:
# No runnable code — answer the questions below based on the model card for
# distilbert-base-uncased-finetuned-sst-2-english
# Card URL: https://huggingface.co/distilbert-base-uncased-finetuned-sst-2-english
# (Read the card, then answer the questions in the markdown cell below)

print('Visit the model card and answer the judgment questions below.')

**Question 5:** For each of the five model card checks, state what you would find for `distilbert-base-uncased-finetuned-sst-2-english` and whether it makes this model appropriate for classifying customer support tickets from a B2B SaaS platform.

1. **Task**: 
2. **Base model**: 
3. **Training data**: 
4. **Evaluation metrics**: 
5. **Licence**: 

**Appropriate for B2B SaaS customer support tickets?** *(Yes/No and one sentence of reasoning.)*

## Summary

> **For each concept, write one sentence on the failure mode it prevents.**

1. `**inputs` dict unpacking in batch inference: 
2. `padding_side = 'left'` for decoder-only models: 
3. Reading the model card before deploying: 